# NanoReason on Kaggle (T4 / P100, 16 GB)

End-to-end run: **baseline -> SFT -> GRPO -> eval -> compare -> export**.

**Before running**, in the right sidebar set:
- **Accelerator** = `GPU T4 x2` (or `GPU P100`)
- **Internet** = `On` (needs a phone-verified Kaggle account)

Uses `configs/kaggle_t4.toml`. All outputs go to `/kaggle/working` so they persist as notebook output.

## 1. Get the code
Upload this repo folder as a Kaggle **Dataset** (any unique title, e.g. `nanoreason-slm`). The cell below finds it under `/kaggle/input/` automatically by searching for `pyproject.toml` — the dataset title and folder nesting don't matter.

In [ ]:
import os, shutil, glob

DST = '/kaggle/working/nanoreason'
if os.path.exists(DST):
    shutil.rmtree(DST)

# Option A: repo uploaded as a Kaggle dataset under /kaggle/input/<name>/
# Search recursively so it works whether the files sit at the dataset root or
# inside a nested 'Ongoing/' folder. Pick the shallowest pyproject.toml.
candidates = glob.glob('/kaggle/input/**/pyproject.toml', recursive=True)
candidates.sort(key=lambda p: p.count('/'))
if candidates:
    src = os.path.dirname(candidates[0])
    shutil.copytree(src, DST)
    print('Copied repo from', src)
else:
    # Option B: clone from GitHub (replace with your URL)
    os.system('git clone https://github.com/<you>/nanoreason.git ' + DST)
    print('Cloned repo to', DST)

os.chdir(DST)
print('cwd =', os.getcwd())
print(os.listdir('.'))

## 2. Install dependencies
Kaggle ships torch + CUDA already; we pin the training libs.

In [ ]:
# Use a single GPU. T4 x2 shards the model across cuda:0/cuda:1 and crashes the
# loss with a device mismatch; Phi-3-mini fits on one 16 GB T4.
%env CUDA_VISIBLE_DEVICES=0

# NOTE: GRPO needs TRL 0.14.x exactly — trl 0.15 changed GRPO batching
# semantics and train_grpo.py refuses to run on >= 0.15.
!pip -q install "transformers>=4.46,<4.47" "trl==0.14.0" "peft>=0.12,<0.14" \
                "datasets>=2.21,<3.0" "accelerate>=0.34,<1.0" "bitsandbytes>=0.43,<0.45"
# --no-deps: every runtime dep was installed above. Without it, pip would try
# to enforce pyproject's torch pin against Kaggle's preinstalled torch and
# could silently download/downgrade a multi-GB CUDA wheel.
!pip -q install -e . --no-deps
# Kaggle's preinstalled torchvision/torchaudio are built for a different torch
# version and break transformers imports ("operator torchvision::nms does not
# exist"). We don't need them for text-LLM training, so remove them.
!pip -q uninstall -y torchvision torchaudio
import torch
print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

## 3. Logic check (no GPU needed)

In [ ]:
!python -m unittest discover -s tests

## 4. Baseline evaluation (subsampled, ~minutes)

In [ ]:
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
!python -m nanoreason.evaluate --config configs/kaggle_t4.toml --run-name baseline

## 5. Stage 2 - Supervised fine-tuning (QLoRA SFT)

In [ ]:
!python -m nanoreason.train_sft --config configs/kaggle_t4.toml

## 6. Stage 3 - GRPO with hybrid PRM/ORM reward
The slow stage. Watch the session clock. Checkpoints land in `/kaggle/working/artifacts/grpo/`.

In [ ]:
!python -m nanoreason.train_grpo --config configs/kaggle_t4.toml

## 7. Evaluate the trained adapter + compare

In [ ]:
!python -m nanoreason.evaluate --config configs/kaggle_t4.toml \
    --adapter /kaggle/working/artifacts/grpo-final --run-name grpo_final
!python -m nanoreason.compare_results \
    --baseline /kaggle/working/results/baseline.json \
    --candidate /kaggle/working/results/grpo_final.json

## 8. Stage 4 - Merge adapter + INT4 inference

In [ ]:
!python -m nanoreason.export --config configs/kaggle_t4.toml \
    --adapter /kaggle/working/artifacts/grpo-final \
    --merged-dir /kaggle/working/artifacts/merged --load-4bit \
    --prompt "A pen costs 3 dollars. How much for 4 pens?"

## Notes
- Session limit ~9-12h. For an unattended full run use **Save Version -> Save & Run All (Commit)**.
- Everything under `/kaggle/working` is saved as output and survives the session.
- The forgetting probe writes `/kaggle/working/results/forgetting_probe.json` during GRPO.
- Session died mid-GRPO? Resume from the last checkpoint:
  `!python -m nanoreason.train_grpo --config configs/kaggle_t4.toml --resume-from-checkpoint /kaggle/working/artifacts/grpo/checkpoint-<N>`
- OOM in GRPO? Lower `num_generations` (4->2) then `max_completion_length` in `configs/kaggle_t4.toml`.